In [0]:
dbutils.widgets.text("business_date", "", "Business Date (YYYY-MM-DD, empty=latest)")

import sys
sys.path.append("/Workspace/Shared")
import aws_utils

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, row_number, substring
)
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()
env = aws_utils.get_env_prefix()

business_date_param = dbutils.widgets.get("business_date").strip()
if business_date_param:
    run_date = business_date_param
else:
    run_date = str(
        spark.table(f"`{env}-cdh`.primary.exposure")
        .filter(col("SourceSystem") == "profits")
        .agg({"BusinessDate": "max"})
        .collect()[0][0]
    )

run_date_lit = lit(run_date).cast("date")
print(f"env: {env}, run_date: {run_date}")

# dtransfer_DebtAB

Replicates on-prem stored procedure `[PLTFRM].[dtransfer].[DebtABsp]`.

Produces one table with loan-party relationships for **Core portfolios only** (Core Retail + Core Wholesale).

## Key Differences from dtransfer_Debt
- dtransfer_Debt covers NON-core portfolios (IsBankingFlag = false)
- dtransfer_DebtAB covers CORE portfolios only (IsBankingFlag = true, PortfolioId 31/32)
- Core portfolios have bare-number ExposureKeys (no prefix)
- Excludes syndicated loan products (koinopraktika)

## Table Mapping

| On-Prem | Cloud | Notes |
| --- | --- | --- |
| `DWH.ast.Asset` (agreement) | `primary.exposure` (AgreementKey) | |
| `DWH.ast.Asset` (loan) | `primary.exposure` / `derivative.dexposure` | ExposureKey |
| `DWH.ast.AssetXPartyRltp` | `primary.exposure` | PartyRelationshipType |
| `DWH.prt.Party` | `primary.party` | PartyId = CDI (agreed cross-platform rule; LEFT SEMI gate on CDI, no DWH PartyId in cloud). IT confirmed the alphanumeric string domain for PartyId is accepted. |
| `DWH.ast.Product` | `primary.product` | ProductKey exclusion |
| `DWH.ref.Portfolio` (31, 32) | `primary.portfolio` (IsBankingFlag = true) | |

## Output

- `{env}-cdh-delivery.it.dtransfer_debtab` (~497K rows)

In [0]:
# ==================== DebtAB ====================
# Core portfolios (IsBankingFlag = true) — PortfolioId 31 (Core Retail), 32 (Core Wholesale)
# Grain: one row per (ExposureKey, CDI, PartyRelationshipType)
# Excludes syndicated loan products (koinopraktika)

# Excluded product codes (syndicated loans)
# On-prem: SUBSTRING(ProductKey, 7, 99) NOT IN (...)
# ProductKey in cloud = 'PortfolioId;ProductCode' e.g. '31;1220'
excluded_product_codes = ['1220','1225','1226','1228','3006','3107','3403','7734','7738','7817']

# Portfolio: core only
portfolio = (
    spark.table(f"`{env}-cdh`.primary.portfolio")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & (col("PortfolioId").isin(31, 32))
    )
    .select("PortfolioId", "PortfolioName")
)

# Exposure: all party relationships for core portfolios, deduplicated
w_exp = Window.partitionBy("ExposureKey", "CDI", "PartyRelationshipType").orderBy(
    when(col("SourceSystem") == "profits", 1).when(col("SourceSystem") == "qca", 2).otherwise(3)
)
exposure = (
    spark.table(f"`{env}-cdh`.primary.exposure")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & (col("IsActiveFlag") == True)
    )
    .withColumn("_rn", row_number().over(w_exp))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

# Party (all-history dedup for PartyId)
# Review #7: run-date scope to match dtransfer_Customer, so DebtStatic.PartyId
# (=CDI) cannot orphan against Customer.PartyId via a historical party row.
party_all = spark.table(f"`{env}-cdh`.primary.party").filter(col("BusinessDate") == run_date_lit)
w_party = Window.partitionBy("CDI").orderBy(
    when(col("SourceSystem") == "profits", 1).when(col("SourceSystem") == "qca", 2).otherwise(3),
    col("BusinessDate").desc()
)
party = (
    party_all.withColumn("_rn", row_number().over(w_party))
    .filter(col("_rn") == 1).drop("_rn")
    .select(col("CDI").alias("pty_CDI"))
)

# Build base: exposure joined to core portfolio
# Base: exposure joined to core portfolio (before the party gate).
debt_ab_base = exposure.join(portfolio, on="PortfolioName", how="inner")

# Party gate = LEFT SEMI on CDI: keeps only exposures whose customer resolves in
# party (drops NULL/unresolvable CDI), matching on-prem. SEMI (not INNER) cannot
# duplicate rows if the party dedup changes.
debt_ab_gated = debt_ab_base.join(party, col("CDI") == col("pty_CDI"), "left_semi")

# Review #12: surface the gate drop; warn (do not block) if larger than ~1%.
_ab_base_cnt = debt_ab_base.count()
_ab_gated_cnt = debt_ab_gated.count()
_ab_dropped = _ab_base_cnt - _ab_gated_cnt
_ab_drop_pct = (_ab_dropped / _ab_base_cnt) if _ab_base_cnt else 0.0
print(f"DebtAB party gate: {_ab_base_cnt:,} -> {_ab_gated_cnt:,} "
      f"(dropped {_ab_dropped:,}, {_ab_drop_pct:.2%})")
if _ab_drop_pct > 0.01:
    print(f"WARNING: party gate dropped {_ab_drop_pct:.2%} of DebtAB rows "
          f"(> 1% threshold) - investigate upstream party/CDI coverage.")

debt_ab = (
    debt_ab_gated
    # PartyId = CDI (agreed rule), taken from the exposure's own CDI.
    .withColumn("PartyId", col("CDI"))
    # Exclude syndicated loan products
    .filter(~col("ProductCode").cast("string").isin(*excluded_product_codes))
    .select(
        run_date_lit.alias("ReferenceDate"),
        col("PortfolioId"),
        col("PartyId"),
        lit(None).cast("long").alias("AgreementId"),
        col("AgreementKey").alias("AgreementNumber"),
        col("ExposureId").alias("LoanId"),
        col("ExposureKey").alias("AccountNumber"),
        col("PartyRelationshipType").alias("AccountRelationType"),
        lit(None).cast("int").alias("AccountRelationTypeCode"),
    )
    .distinct()
)

# Review #14: PartyId (=CDI) must be non-null on every row.
assert debt_ab.filter(col("PartyId").isNull()).limit(1).count() == 0, \
    "DebtAB has NULL PartyId rows (PartyId=CDI invariant violated)"

print(f"DebtAB rows: {debt_ab.count():,}")

In [0]:
# ==================== Write DebtAB ====================
target = f"`{env}-cdh-delivery`.it.dtransfer_debtab"
debt_ab.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
print(f"Written to {target}")